In [16]:
import pandas as pd

df = pd.read_parquet("../data/clean_yellow_tripdata_2025-01.parquet")

In [17]:
# Duration in minutes
df['trip_duration_min'] = (df['tpep_dropoff_datetime'] - df['tpep_pickup_datetime']).dt.total_seconds()/60

# Tip as percentage of fare
df['tip_pct'] = df['tip_amount'] / df['total_amount'] * 100
df.fillna({'tip_pct':0}, inplace=True)

# remove impossible trips
df = df[
    (df['trip_duration_min'] > 0) &
    (df['trip_distance'] > 0) &
    (df['total_amount'] > 0)
]

# Trip speed (mph)
df['trip_speed_mph'] = df['trip_distance'] / (df['trip_duration_min']/60)
df.replace([float('inf'), -float('inf')], 0, inplace=True)
df.fillna({'trip_speed_mph': 0}, inplace=True)

# Revenue per mile
df['revenue_per_mile'] = df['total_amount'] / df['trip_distance']
df.replace([float('inf'), -float('inf')], 0, inplace=True)
df.fillna({'revenue_per_mile': 0}, inplace=True)

# Flags
df['long_trip'] = df['trip_distance'] > 10
df['high_tip'] = df['tip_pct'] > 20

In [18]:
print(df['trip_duration_min'].describe())
print(df['tip_pct'].describe())
print(df['trip_speed_mph'].describe())

count    737.000000
mean       1.421280
std        3.433970
min        0.016667
25%        0.100000
50%        0.266667
75%        0.616667
max       20.766667
Name: trip_duration_min, dtype: float64
count    737.000000
mean       5.198193
std       12.370741
min        0.000000
25%        0.000000
50%        0.000000
75%        3.636364
max       95.639944
Name: tip_pct, dtype: float64
count      737.000000
mean      1793.559988
std       4594.729063
min        100.327869
25%        211.017639
50%        520.000000
75%       1602.000000
max      67608.000000
Name: trip_speed_mph, dtype: float64


In [19]:
# Keep only trips with reasonable duration and distance

df = df[
    (df['trip_duration_min'] > 1) &      
    (df['trip_duration_min'] < 120) &   
    (df['trip_distance'] > 0) &         
    (df['total_amount'] > 0)            
]

In [20]:
tip_upper = df['tip_pct'].quantile(0.99)
df = df[df['tip_pct'] <= tip_upper]

In [23]:
df['trip_speed_mph'] = df['trip_speed_mph'].clip(0, 80)

In [24]:
print(df[['trip_duration_min','tip_pct','trip_speed_mph']].describe())

       trip_duration_min     tip_pct  trip_speed_mph
count         111.000000  111.000000           111.0
mean            7.863063    2.862281            80.0
std             5.411424    6.271312             0.0
min             1.016667    0.000000            80.0
25%             3.083333    0.000000            80.0
50%             6.683333    0.000000            80.0
75%            11.991667    0.000000            80.0
max            20.766667   28.037383            80.0


In [25]:
revenue_hour = df.groupby('pickup_hour')['total_amount'].sum().reset_index()
revenue_hour.to_csv("../data/revenue_by_hour.csv", index=False)

trips_hour = df.groupby('pickup_hour').size().reset_index(name='num_trips')
trips_hour.to_csv("../data/trips_by_hour.csv", index=False)

In [26]:
vendor_kpis = df.groupby('VendorID').agg(
    total_revenue = ('total_amount', 'sum'),
    avg_revenue = ('total_amount', 'mean'),
    num_trips = ('total_amount', 'count'),
    avg_tip_pct = ('tip_pct', 'mean'),
    long_trip_pct = ('long_trip', 'mean')
).reset_index()

vendor_kpis['long_trip_pct'] *= 100
vendor_kpis.to_csv("../data/vendor_kpis.csv", index=False)

In [27]:
zone_kpis = df.groupby('PULocationID').agg(
    avg_distance=('trip_distance', 'mean'),
    long_trip_ratio=('long_trip', 'mean'),
    avg_tip_pct=('tip_pct', 'mean'),
    num_trips=('trip_distance', 'count')
).reset_index()

zone_kpis['long_trip_ratio'] *= 100
zone_kpis.to_csv("../data/zone_kpis.csv", index=False)

In [28]:
weekday_kpis = df.groupby('is_weekend').agg(
    avg_total=('total_amount', 'mean'),
    avg_distance=('trip_distance', 'mean'),
    avg_tip_pct=('tip_pct', 'mean'),
    num_trips=('total_amount', 'count')
).reset_index()

weekday_kpis.to_csv("../data/weekday_vs_weekend.csv", index=False)

In [29]:
df.to_parquet("../data/clean_yellow_tripdata_2025-01.parquet", index=False)
df.to_csv("../data/clean_yellow_tripdata_2025-01.csv", index=False)